In [ ]:
import math
import multiprocessing
from multiprocessing import Pool
# import torch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import fnmatch
import sys

import joblib

from tqdm.notebook import tqdm


In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
import math

In [ ]:

from itertools import combinations
import time
from shapely.ops import unary_union
from shapely.geometry import Point
from shapely.geometry import Point, Polygon
from functools import reduce
import pickle

In [ ]:
from tqdm.notebook import tqdm

In [ ]:
import seaborn as sns
from collections import defaultdict
from collections import Counter

In [ ]:
pwd

In [ ]:
%run ./common_vars.py
%run ~/repos/spatialtuning/funs/plotting.py
%run ~/repos/spatialtuning/funs/utils.py

In [ ]:
meta = pd.read_csv("/glade/work/qingyuany/repo_data/spatialtuning/meta_cli_para.csv", index_col = 0)
tf_masks_raw = pd.read_csv("/glade/work/qingyuany/repo_data/spatialtuning/zonal_masks_2.0.csv", index_col=0)
para_emu = xr.open_dataset("/glade/work/qingyuany/repo_data/spatialtuning/para_samples.nc").to_dataframe()
para_nm = list(para_emu.columns)

## Filter variables to ignore

In [ ]:
polar_vars = [localvar for localvar in list(tf_masks_raw.columns) if ((localvar.split("_")[1] == "zonal") | 
                                                        (localvar.split("_")[2] == "zonal")) & 
                                                        (abs(float((localvar.split("_")[-1]))) > 75) ]

useless_vars = list(tf_masks_raw.columns[tf_masks_raw.sum(axis = 0) == 1000000])
manual_remove_vars = list(tf_masks_raw.columns[tf_masks_raw.columns.str.startswith("CLDTOT_ISCCP_zonal")]) + list(tf_masks_raw.columns[tf_masks_raw.columns.str.startswith("TGCLDLWP_zonal")])

In [ ]:
ignore_vars = polar_vars + useless_vars + manual_remove_vars

In [ ]:
tf_masks_raw = tf_masks_raw.drop(columns = ignore_vars, errors = "ignore")#

In [ ]:
strt_threshold = 100000
strt_vars = list(tf_masks.columns[((tf_masks.sum(axis = 0) < strt_threshold))])

In [ ]:
tf_masks_raw = tf_masks_raw.drop(columns = strt_vars)


## Group the and filter more variables based on emulator meta data

In [ ]:
rm_3d_var = ['LWCF_zonal_lat_-67', 'FLUT_zonal_lat_-57', 'FLUT_zonal_lat_-52', 'LWCF_zonal_lat_-67', 
             'FLUT_zonal_lat_-62','FLUT_zonal_lat_57', 'LWCF_zonal_lat_37', 'LWCF_zonal_lat_32', 
             'FLUT_zonal_lat_52', 'LWCF_zonal_lat_72', 'TMQ_zonal_lat_-27', 'TMQ_zonal_lat_-22',
            'FLUT_zonal_lat_-42', 'FSNTOA_zonal_lat_-67', 'SWCF_zonal_lat_-52', 'SWCF_zonal_lat_-57', 
             'LWCF_zonal_lat_-32', 'LWCF_zonal_lat_-27', 
            'FLUT_zonal_lat_-47', 'SWCF_zonal_lat_-17']

tf_masks = tf_masks_raw.drop(columns = rm_3d_var)

In [ ]:
paras_vars, strt_paras_vars, surv_paras_vars = group_para_climatology(para_emu, tf_masks, meta, threshold = 100000)

In [ ]:
strterr_sum =  strterr_detection(strt_paras_vars, tf_masks, n_comb = 2)

In [ ]:
list(strterr_sum.values())[0]

In [ ]:
para1_vars_dict, para1_para2_dict, para1_ranges = range_err_detection2(surv_paras_vars, tf_masks, para_emu, meta)

In [ ]:
a = para1_error_localvar_detection(para1_vars_dict, "microp_aero_wsubi_scale", para_emu, tf_masks, n_comb = 1)

In [ ]:
para2_vars, para2_vars_s, para2_vars_area, para2_hulls = para2_error_detection(surv_paras_vars, tf_masks, para_emu, meta, shape_alpha = 5)

In [ ]:
para2_area = para2_error_localvar_detection(para2_vars, ['clubb_C8', 'microp_aero_wsubi_scale'], para_emu,tf_masks, n_comb = 1,  shape_alpha = 7)

In [ ]:
para2_ratio, para2_hulls = ratio_cal2d(para2_hulls)

In [ ]:
with open('/glade/work/qingyuany/repo_data/spatialtuning/python_obj/hulls_2d.pkl', 'wb') as f:
    pickle.dump(para2_hulls, f)

In [ ]:
#para3_hulls = para3_mesh_generator(paras_vars, para_emu, tf_masks)

In [ ]:
#para3_ratio, para3_hulls = ratio_cal3d(para3_hulls)

In [ ]:
#with open('/glade/work/qingyuany/repo_data/spatialtuning/python_obj/hulls_3d.pkl', 'wb') as f:
#    pickle.dump(para3_hulls, f)

## Sampling

In [ ]:
para2_group_summary(para2_hulls)


In [ ]:
hulls_a = group_by_para(para2_hulls, 'clubb_C8')
hulls_b = group_by_para(para2_hulls, 'microp_aero_wsubi_scale')
hulls_c = group_by_para(para2_hulls, 'clubb_c2rt')
hulls_rest = group_by_rest(para2_hulls, [hulls_a, hulls_b, hulls_c])

In [ ]:
temp_pts = sample2d(2000000, para_nm, para1_ranges, hulls_a, existing_pts = np.NAN, monitor = True)

In [ ]:
sample2d(2000000, para_nm, para1_ranges, hulls_b, existing_pts = temp_pts, monitor = True)

In [ ]:
sample2d(2000000, para_nm, para1_ranges, hulls_c, existing_pts = np.NAN, monitor = True)

In [ ]:
sample2d(2000000, para_nm, para1_ranges, hulls_rest, existing_pts = np.NAN, monitor = True)

In [ ]:
final_samples = []
for i in np.arange(5):
    aa = sample2d_perlist(hulls_a, para_nm, para1_ranges, 5, exist_pts = np.nan, n_pts= 1000000, n_cpu = 30, n_need = 2000000)
    bb = sample2d_wrapper(hulls_b, para_nm, para1_ranges, aa)
    cc  = sample2d_wrapper(hulls_c, para_nm, para1_ranges, bb)
    dd  = sample2d_wrapper(hulls_rest, para_nm, para1_ranges, cc)
    final_samples.append(dd)

In [ ]:
final_para = pd.concat(final_samples, axis  = 0)

In [ ]:
plt.scatter(aa.clubb_C8, aa.microp_aero_wsubi_scale)
plt.scatter(final_para.clubb_C8, final_para.microp_aero_wsubi_scale)

In [ ]:
plt.scatter(bb.clubb_C8, bb.microp_aero_wsubi_scale)